[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day05-transformer-block.ipynb)
# Day 5 — The Transformer Block: Pre-Norm, SwiGLU, Residuals
Assemble the full decoder block, audit where the parameters live, and measure gradient health across depth. CPU-OK, ~30–45 min.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
# Expected: installs quietly, no output above this line

## 1. RMSNorm
`x / sqrt(mean(x^2) + eps) * gamma` — LayerNorm without the mean-centering step. Per-row RMS comes out ≈ 1 before the learned gain (gamma inits to 1).

In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))   # gamma
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

torch.manual_seed(0)
norm = RMSNorm(512)
x = torch.randn(2, 64, 512) * 5          # deliberately off-scale input
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
print("per-row RMS: min %.4f  max %.4f" % (rms.min(), rms.max()))
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print("RMSNorm check passed: per-row RMS ≈ 1.0  (shape:", tuple(y.shape), ")")
# Expected: per-row RMS: min 1.0000  max 1.0000 / check passed

## 2. SwiGLU MLP
`down(silu(gate(x)) * up(x))` — three matrices. Toy hidden width 1365 ≈ 8/3 × 512.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model=512, d_ff=1365):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False)
        self.up   = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.down(nn.functional.silu(self.gate(x)) * self.up(x))

mlp = SwiGLU()
n = sum(p.numel() for p in mlp.parameters())
print(f"SwiGLU(512→1365→512) params: {n:,}  (= 3 × 512 × 1365 = {3*512*1365:,})")
assert n == 3 * 512 * 1365
# Expected: SwiGLU(512→1365→512) params: 2,099,520

## 3. The decoder block
Pre-norm assembly: `x + attn(norm1(x))`, then `x + mlp(norm2(x))`. A compact causal MHA with RoPE is included so this notebook is self-contained (same math as Day 4).

In [ ]:
import math

def rope_tables(seq_len, d, base=10_000.0):
    freqs = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    t = torch.arange(seq_len).float()
    ang = torch.outer(t, freqs)
    return torch.cos(ang), torch.sin(ang)          # (seq, d/2)

def apply_rope(x, cos, sin):
    # x: (B, heads, seq, head_dim); rotate pairs (2i, 2i+1)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    c, s = cos[:x.shape[2]].unsqueeze(0).unsqueeze(0), sin[:x.shape[2]].unsqueeze(0).unsqueeze(0)
    return torch.stack([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).flatten(-2)

class MHA(nn.Module):
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.hd = n_heads, d_model // n_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B, T, _ = x.shape
        q = self.Wq(x).view(B, T, self.n_heads, self.hd).transpose(1, 2)
        k = self.Wk(x).view(B, T, self.n_heads, self.hd).transpose(1, 2)
        v = self.Wv(x).view(B, T, self.n_heads, self.hd).transpose(1, 2)
        cos, sin = rope_tables(T, self.hd)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.Wo(y.transpose(1, 2).reshape(B, T, -1))

class DecoderBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=1365):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = MHA(d_model, n_heads)
        self.mlp = SwiGLU(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))     # pre-norm: stream untouched
        x = x + self.mlp(self.norm2(x))
        return x

torch.manual_seed(1)
block = DecoderBlock()
x = torch.randn(2, 128, 512)
y = block(x)
print("Block shape-preserving:", tuple(x.shape), "->", tuple(y.shape))
assert y.shape == x.shape
# Expected: Block shape-preserving: (2, 128, 512) -> (2, 128, 512)

## 4. Parameter audit — where do the params live?
Expect: attn = 4 × 512² = 1,048,576; MLP ≈ 2.0× that at exact 8/3 parity. Then the Llama-3.1-8B audit in pure arithmetic.

In [ ]:
attn_n = sum(p.numel() for p in block.attn.parameters())
mlp_n  = sum(p.numel() for p in block.mlp.parameters())
norm_n = sum(p.numel() for p in list(block.norm1.parameters()) + list(block.norm2.parameters()))
print(f"attention: {attn_n:,}  |  MLP: {mlp_n:,}  |  norms: {norm_n:,}")
print(f"MLP / attention ratio: {mlp_n/attn_n:.2f}x")
assert attn_n == 4 * 512 * 512

# Llama-3.1-8B: d=4096, d_ff=14336, 32 layers (no download — pure arithmetic)
d, d_ff, L = 4096, 14336, 32
mlp_layer, attn_layer = 3*d*d_ff, 4*d*d
mlp_tot, attn_tot = mlp_layer*L, attn_layer*L
print(f"\nLlama-3.1-8B per layer: MLP {mlp_layer/1e6:.1f}M vs attention {attn_layer/1e6:.1f}M "
      f"(ratio {mlp_layer/attn_layer:.2f}x)")
print(f"32 layers: MLP {mlp_tot/1e9:.2f}B ({mlp_tot/8.03e9:.1%} of 8.03B) | "
      f"attention {attn_tot/1e9:.2f}B")
# Expected: attention: 1,048,576 | MLP: 2,099,520 | ratio 2.00x
#           Llama-3.1-8B per layer: MLP 176.2M vs attention 67.1M (ratio 2.63x)
#           32 layers: MLP 5.64B (70.2% of 8.03B) | attention 2.15B

## 5. Gradient health: pre-norm across 8 blocks
Stack 8 blocks, backprop, print the gradient norm at each block's input. Healthy = all within ~one order of magnitude, no collapse toward the input.

In [ ]:
def grad_norms(make_block, n_blocks=8):
    torch.manual_seed(7)
    blocks = nn.Sequential(*[make_block() for _ in range(n_blocks)])
    x = torch.randn(2, 64, 512, requires_grad=True)
    h, inputs = x, [x]
    for b in blocks:                       # full graph, no detach — real depth
        h = b(h)
        h.retain_grad()
        inputs.append(h)
    h.sum().backward()
    return [t.grad.norm().item() for t in inputs[:-1]]   # grad at each block's input

pre = grad_norms(lambda: DecoderBlock())
for i, g in enumerate(pre):
    print(f"block {i}: grad norm {g:.3f}")
print("max/min ratio: %.1fx" % (max(pre)/min(pre)))
assert max(pre)/min(pre) < 10, "pre-norm gradients should stay O(1)"
print("Pre-norm: gradients healthy across depth")
# Expected: 8 lines of grad norms all within ~10x of each other, e.g. 0.8 … 2.3

## 6. The 2017 problem: post-norm comparison
Same stack, but `norm(x + sublayer(x))`. Watch the early-layer gradients decay.

In [ ]:
class PostNormBlock(nn.Module):
    def __init__(self, d_model=512, n_heads=8, d_ff=1365):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(d_model), RMSNorm(d_model)
        self.attn = MHA(d_model, n_heads)
        self.mlp = SwiGLU(d_model, d_ff)
    def forward(self, x):
        x = self.norm1(x + self.attn(x))     # post-norm: stream renormalized each layer
        x = self.norm2(x + self.mlp(x))
        return x

post = grad_norms(lambda: PostNormBlock())
for i, g in enumerate(post):
    print(f"block {i}: grad norm {g:.3f}")
print("max/min ratio: %.1fx" % (max(post)/min(post)))
print("Post-norm: early layers starved — this is why nobody ships post-norm past ~20 layers")
# Expected: block-0 grad norm an order of magnitude (or more) below block-7

## Wrap-up
- A decoder block = RMSNorm → MHA(+RoPE) → residual → RMSNorm → SwiGLU → residual.
- Pre-norm keeps the residual stream unnormalized: clean identity gradient path, trains 100+ layers deep.
- Llama-3.1-8B: 5.64B of 8.03B params (70%) live in the SwiGLU MLPs — quantization and MoE target the MLP; KV work targets attention.
- Tomorrow (Day 6): the parts become a whole — a complete GPT in ~150 lines, trained on Tiny Shakespeare.